# D50: FULL 90 câu bằng **ReAct + Calculator tool** (Qwen3-4B)

Dạng D50: cho $|z-z_1|+|z-z_2|=K$ (với $K=AB$, nên $M$ chạy trên đoạn
thẳng $AB$ theo bất đẳng thức tam giác); tìm GTNN $m$, GTLN $M$ của
$|z-z_3|$, tính $P=m+M$. Phương pháp: hình học giải tích (đường thẳng
$AB$) kết hợp khảo sát hàm số bậc hai $f(x)=MC^2$ trên đoạn $[\min(x_A,x_B),
\max(x_A,x_B)]$. Bản chạy đầy đủ của phương pháp đã kiểm chứng tay 11 câu
trong `KLTN_D50_ReAct_Calculator_1cau.ipynb` (câu gốc + 10 câu khó nhất,
đa dạng cả 3 loại số, bao gồm nhiều câu thuộc nhóm "$x_0$ nằm ngoài đoạn"
- nhóm phổ biến nhất, 55/91 câu).

Model **không tự tính tay bất kỳ phép nào**: mọi phép tính đều gọi tool
`Calculator` (sympy, chính xác tuyệt đối, **có bộ nhớ biến**, dùng
`radsimp()` làm bước rút gọn chính để tránh treo máy với mẫu số chứa
nhiều căn khác gốc) theo vòng lặp ReAct thật:
`Thought → Action → Action Input → Observation → …`

Prompt được **ép mở đầu bằng `<think>\nThought:`** (forced prefix);
model không còn quyền tự chọn viết văn xuôi mở đầu trước khi vào định dạng
ReAct.

## Điểm khác bản 1 câu: vòng lặp ReAct chạy THEO LÔ

Chạy tuần tự 90 câu sẽ mất hàng giờ. Ở đây mỗi **vòng** gọi vLLM **một lần**
cho tất cả các câu đang hoạt động (vLLM tự batching), rồi chạy Calculator
riêng cho từng câu, rồi generate tiếp.

Mỗi câu có **bộ nhớ biến riêng** (`MayTinh()` riêng), **ngân sách token
riêng**, và tự thoát khỏi lô khi viết xong `Final Answer`.

Hạn mức tool gọi/câu (`50          # quy trinh day du ~18-19 luot tinh + toi da 4 luot doi chieu phuong an; nang tu 35 vi cau STT1069 (truong hop duong thang dung, hiem gap, chua co vi du mau) dung het 35 luot voi 25/35 la loi cu phap - can du cho nhieu lan thu-sai hon`) đủ dư cho khoảng 27-29 lượt tính
đầy đủ quy trình cộng tối đa 4 lượt so khớp phương án.

## Prompt & backend giống hệt bản 1 câu

Cell 4 (prompt) và phần backend `MayTinh` ở Cell 5 được **trích nguyên văn**
từ notebook 1 câu bằng script `scratch/build_react_full_D50.py`; không gõ
lại, nên không có nguy cơ lệch giữa 2 bản.

## Trước khi chạy

Upload `plan_solve_prompts_D50.json` (90 câu, đã sinh sẵn từ
`Sinh_them_cau_hoi/So_phuc_day_du.csv`, đã loại câu gốc STT 12 nằm trong
few-shot) thành Kaggle Dataset (slug gợi ý `d50-full90`), gắn vào notebook,
bật GPU.

Kết quả: `/kaggle/working/d50_react_calculator_full90.csv`

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

DATA_PATH = '/kaggle/input/d50-full90/plan_solve_prompts_D50.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d50_react_calculator_full90.csv'

MODEL = 'Qwen/Qwen3-4B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = set()

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D50_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision - this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True` - test all four, every time (see PART 4's matching step for why).
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo - retyping a small variation of the same idea will keep producing the same error. Stop, go back to PART 2's method for this exact situation (a formula giving `zoo`/an undefined result almost always means a special case described somewhere in PART 2 applies here - e.g. a division that is only valid when some quantity is nonzero), and use the alternative formula PART 2 gives for that case, typed as a normal exact expression - never invent a placeholder word like `undefined` as if it were a value; the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================
# [B] PART_KIENTHUC - KIEN THUC NEN SO PHUC (DUNG CHUNG MOI DANG SO PHUC)
# =====================================================================
PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$ - there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change - so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
'''


# =====================================================================
# [C] PART_HUONGGIAI - HUONG GIAI RIENG CUA DANG D50 (THAY KHI DOI DANG)
# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE (this is a worked-through analysis, not a checklist - follow the reasoning itself, so you can reconstruct the right approach for any numbers, including cases that look nothing like the worked example in PART 3)**

Problem shape: $|z-z_1|+|z-z_2|=K$ (given, $K$ a positive real constant). Find the min $m$ and max $M$ of $|z-z_3|$, then compute $P=m+M$ (or whatever combination the problem asks).

**Turning the algebra into geometry.** Let $M,A,B,C$ be the points representing $z,z_1,z_2,z_3$. By fact 17 (PART 1), $|z-z_1|=MA$ and $|z-z_2|=MB$, so the given equation is exactly $MA+MB=K$: as $z$ ranges over every complex number satisfying it, $M$ ranges over every point whose distances to two FIXED points $A,B$ sum to the FIXED constant $K$. Read $A,B,C$ off the problem term by term: each modulus term $|z-w|$ needs $w$ isolated from the part written after $z$, by negating that part coefficient by coefficient (real part and imaginary part separately); the corresponding point is $(\mathrm{Re}(w),\mathrm{Im}(w))$.

**Why $M$ turns out to be confined to the segment $AB$, and nowhere else.** Compute $AB=\sqrt{(x_B-x_A)^2+(y_B-y_A)^2}$ (fact 17) and check whether it equals the given $K$. This single check is the crux of the whole problem, and it is worth seeing exactly why: fact 18 says $MA+MB\ge AB$ for ANY point $M$ in the plane, with equality precisely when $M$ lies on the segment between $A$ and $B$. So if $AB=K$, the condition $MA+MB=K$ is EXACTLY the equality case of fact 18 - it is not merely \"some distance condition\", it pins $M$ down to the finite segment $AB$, and every point of that segment satisfies it. This is what turns an optimization problem over the entire plane into an optimization problem over a 1-dimensional segment - without it, none of what follows would be tractable.

**Trying to describe where $M$ is on the segment - and discovering there are two cases, not one.** To search over the segment, we need a way to name \"the point at such-and-such position along $AB$\". The natural attempt is fact 19: write the line through $A,B$ as $y=a(x-x_A)+y_A$ with slope $a=\dfrac{y_B-y_A}{x_B-x_A}$, and let $x$ range over $[\min(x_A,x_B),\max(x_A,x_B)]$. But fact 19 also says this attempt does not always work: if $x_A=x_B$, there is no such slope (the Calculator will report `zoo` when you try), precisely because the segment $AB$ is VERTICAL - $x$ equals $x_A$ at every point of it, and it is $y$ that ranges over $[\min(y_A,y_B),\max(y_A,y_B)]$ instead. So the very first thing to do is compute $a=\dfrac{y_B-y_A}{x_B-x_A}$ and look at what comes back, BEFORE assuming which case applies:
  - an ordinary number $\Rightarrow$ the **main case** below;
  - `zoo` $\Rightarrow$ the **vertical case** below. Do not retry the same formula, and do not invent a placeholder value such as \"undefined\" - the Calculator has no such value, only exact numbers. Skip straight to the vertical case, which reuses every argument from the main case with $x$ and $y$ swapped (fact 19's own symmetry).

**Main case ($x_A\ne x_B$) - building $MC^2$ as a function of position on the segment.** With $y=a(x-x_A)+y_A$ established, and writing $k=y_A-y_C$ to shorten the algebra, substituting into $MC^2=(x-x_C)^2+(y-y_C)^2$ gives $f(x)=(x-x_C)^2+[a(x-x_A)+k]^2$. Two deliberate choices here both serve the same goal - keeping the algebra from exploding once $x_A,y_A,\dots$ contain several different square roots:
  - never compute the $y$-intercept $b=y_A-a\,x_A$ as its own stored quantity - point-slope form already fully describes the same line, and forming $b$ separately would mean carrying a whole extra fraction through everything that follows, for no benefit;
  - never expand $f(x)$ into a polynomial $Ax^2+Bx+C$ - every term of that expansion would carry cross-products of whatever radicals are inside $a,x_A,x_C,k$, which is exactly what makes the algebra explode in hard problems. The squared form above is mathematically identical and stays short - leave it exactly as is.

**Why $f$ has exactly one lowest point, and how to find it without `solve()`.** $f(x)$ is a sum of two squared real quantities, each linear in $x$; expanded, its $x^2$ coefficient would be $1+a^2$, always positive. So $f$ is an upward parabola: it has one lowest point (its vertex) and rises without bound moving away from it in either direction - nowhere else can its slope be zero. Differentiate with the chain rule, $f'(x)=2(x-x_C)+2a[a(x-x_A)+k]$, set it to $0$, and solve this LINEAR equation for $x$ by hand to get the closed form once and for all: $x_0=\dfrac{x_C+a^2x_A-ak}{a^2+1}$. Since this formula already IS the solved result, compute it with ONE Calculator call - calling `diff()`/`solve()` would mean solving something already solved.

**Why $x_0$ only sometimes matters.** $x_0$ minimizes $f$ over the WHOLE real line, but $M$ can only be at points of the segment, $x\in[\min(x_A,x_B),\max(x_A,x_B)]$, a strict piece of that line in general. Two genuinely different situations follow from where $x_0$ lands:
  - if $x_0$ is inside that interval, $M$ can actually reach the true lowest point of $f$, so $f(x_0)$ is a real candidate for the minimum; and since $f$ keeps rising moving away from $x_0$ in either direction, whichever endpoint is FARTHER from $x_0$ gives the largest value on the interval, so both endpoints still matter too;
  - if $x_0$ is outside that interval, the ENTIRE interval lies on one side of the vertex, where $f$ is purely increasing or purely decreasing across the whole thing (the only place its slope changes sign is at $x_0$, which is not here). A monotonic function on a closed interval attains both its minimum and maximum at the two endpoints, full stop - $x_0$ is irrelevant and must be discarded, not used for either $m$ or $M$.
This is a single yes/no question, not two separate facts to recall and combine afterwards in your head - trying to remember two `True`/`False` results and reason about them in words is exactly where mistakes creep in. There is a one-shot arithmetic test instead: for $t$ between bounds $\mathrm{lo},\mathrm{hi}$, both $(t-\mathrm{lo})$ and $(\mathrm{hi}-t)$ are non-negative, so their PRODUCT is non-negative; outside either bound, exactly one factor is negative, making the product negative. Compute `inside_segment = (x0 - Min(Ax,Bx)) * (Max(Ax,Bx) - x0) >= 0` as ONE Calculator call and just read the answer - never re-derive or restate it from memory afterwards. Being outside is actually the MORE common situation across this problem family (roughly half of all cases) - never default to assuming $x_0$ is inside.

**The candidates (main case).** The two endpoints are always candidates and are always cheap: at $x=x_A$, $M$ coincides with $A$ itself, so $f(x_A)$ is simply $AC^2=(x_A-x_C)^2+(y_A-y_C)^2$ (plain distance, fact 17) - never substitute $x_A$ into the line/$f(x)$ formulas, that drags in $a,k$ for nothing. Likewise $f(x_B)=BC^2$. If `inside_segment` was `True`, also compute $f(x_0)=(x_0-x_C)^2+[a(x_0-x_A)+k]^2$ using the still-unexpanded form; if `False`, do not compute this at all - it does not correspond to anywhere $M$ can actually be. Take the min and max among the surviving candidates; since each is a squared distance it is automatically $\ge0$, so the smallest is $m^2$ and the largest is $M^2$; square-root them for $m,M$.

**Vertical case ($x_A=x_B$) - the exact same reasoning, with $x$ and $y$ swapped.** Here $x$ is fixed at $x_A$ everywhere on the segment, and it is $y$ that ranges over $[\min(y_A,y_B),\max(y_A,y_B)]$. Nothing new needs to be invented; every argument above carries over verbatim under this swap. $MC^2$ as a function of $y$ is $g(y)=(x_A-x_C)^2+(y-y_C)^2$, itself an upward parabola, this time in $y$, whose vertex is at $y=y_C$ exactly (there is no analog of $a,k$ to carry, since $x$ is simply constant here - do not try to compute them in this branch). \"Is the vertex inside the interval\" becomes the same product test, with $y$'s instead of $x$'s: `inside_segment = (Cy - Min(Ay,By)) * (Max(Ay,By) - Cy) >= 0`. The two endpoints are $AC^2,BC^2$ exactly as before (fact 17 never depended on which coordinate did the work). If `inside_segment` is `True`, the third candidate is $g(y_C)=(x_A-x_C)^2+(y_C-y_C)^2=(x_A-x_C)^2$ - simply `f_x0 = (Ax-Cx)**2`, since the $(y-y_C)^2$ term vanishes exactly at $y=y_C$.

**Compute and match.** Compute whatever combination of $m,M$ the problem asks for (e.g. $P=m+M$). As explained in the Calculator instructions above, `Eq()` here is a high-precision numeric check, not a symbolic proof - so test ALL FOUR options every time, one at a time in order, never stopping early even once one comes back `True`; exactly one of the four should return `True`, and that letter is the answer.
'''

# [D] PART_FEWSHOT - VI DU MAU RIENG CUA DANG D50 (THAY KHI DOI DANG)
# Cac dong "Observation:" duoi day la KET QUA THAT (chay bang chinh class
# MayTinh o backend, khong phai sympy roi rac va khong phai so bia).
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Xét các số phức $z$ thỏa mãn $|z+2-i|+|z-4-7i| = 6\sqrt{2}$. Gọi $m, M$ lần lượt là giá trị nhỏ nhất, giá trị lớn nhất của $|z-1+i|$. Tính $P = m+M$.
A. $P=\sqrt{13}+\sqrt{73}$ | B. $P=\dfrac{5\sqrt{2}+2\sqrt{73}}{2}$ | C. $P=5\sqrt{2}+\sqrt{73}$ | D. $P=\dfrac{5\sqrt{2}+\sqrt{73}}{2}$

<think>
Thought: The first term is |z+2-i|. The part right after z is +2-i, so its real part is +2 and its imaginary part is -1. Since the term equals z-w, w must be the negative of that part: w=-(2-i). Distributing the minus sign carefully: 2 becomes -2, and -i becomes +i. So w=-2+i: its real part is -2 and its imaginary part is 1, and those become A's x and y exactly as they are, with no further sign change. So A=(-2,1). I will now type exactly: Ax = -2
Action: Calculator
Action Input: Ax = -2
Observation: Ax = -2

Thought: Store the y-coordinate of A too; never skip it. I will now type exactly: Ay = 1
Action: Calculator
Action Input: Ay = 1
Observation: Ay = 1

Thought: The second term is |z-4-7i|. The part right after z is -4-7i, so its real part is -4 and its imaginary part is -7. Since the term equals z-w, w must be the negative of that part: w=-(-4-7i). Distributing the minus sign carefully: -4 becomes +4, and -7i becomes +7i. So w=4+7i: its real part is 4 and its imaginary part is 7, and those become B's x and y exactly as they are, with no further sign change. So B=(4,7). I will now type exactly: Bx = 4
Action: Calculator
Action Input: Bx = 4
Observation: Bx = 4

Thought: Store the y-coordinate of B too; never skip it. I will now type exactly: By = 7
Action: Calculator
Action Input: By = 7
Observation: By = 7

Thought: The target term is |z-1+i|. The part right after z is -1+i, so its real part is -1 and its imaginary part is +1. Since the term equals z-w, w must be the negative of that part: w=-(-1+i). Distributing the minus sign carefully: -1 becomes +1, and +i becomes -i. So w=1-i: its real part is 1 and its imaginary part is -1, and those become C's x and y exactly as they are, with no further sign change. So C=(1,-1). I will now type exactly: Cx = 1
Action: Calculator
Action Input: Cx = 1
Observation: Cx = 1

Thought: Store the y-coordinate of C too; never skip it. I will now type exactly: Cy = -1
Action: Calculator
Action Input: Cy = -1
Observation: Cy = -1

Thought: Store K too. I will now type exactly: K = 6*sqrt(2)
Action: Calculator
Action Input: K = 6*sqrt(2)
Observation: K = 6*sqrt(2)

Thought: Check AB=K.
Action: Calculator
Action Input: AB = sqrt((Bx-Ax)**2 + (By-Ay)**2)
Observation: AB = 6*sqrt(2)

Action: Calculator
Action Input: Eq(AB, K)
Observation: True

Thought: Confirmed, M lies on segment AB. Find only the slope a - I never compute the y-intercept b, since that is where huge fractions come from in hard problems. The line is y=a(x-Ax)+Ay.
Action: Calculator
Action Input: a = (By-Ay)/(Bx-Ax)
Observation: a = 1

Thought: That is the exact value of a, straight from the Calculator, so it is correct; I never re-derive or double-check it by hand, whatever it looks like.
Action: Calculator
Action Input: k = Ay - Cy
Observation: k = 2

Thought: f(x)=(x-Cx)^2+[a(x-Ax)+k]^2 stays in this squared form; I never expand it into a polynomial, since that is exactly what makes hard problems explode. Its derivative set to zero solves directly to the closed form x0=(Cx+a^2*Ax-a*k)/(a^2+1); I use this formula straight away instead of calling diff()/solve().
Action: Calculator
Action Input: x0 = (Cx + a**2*Ax - a*k)/(a**2+1)
Observation: x0 = -3/2

Thought: To check whether x0 lies inside the segment [min(Ax,Bx), max(Ax,Bx)] = [-2,4] in one shot, I use the product-of-differences trick: (x0-min)*(max-x0) is non-negative exactly when x0 is between them.
Action: Calculator
Action Input: inside_segment = (x0 - Min(Ax, Bx)) * (Max(Ax, Bx) - x0) >= 0
Observation: inside_segment = True

Thought: inside_segment is True, so x0 lies inside the segment and is a valid third candidate. Now the two endpoints: since M=A at the left endpoint, f there is simply AC^2, computed directly from coordinates - no need to substitute into f(x).
Action: Calculator
Action Input: AC2 = (Ax-Cx)**2 + (Ay-Cy)**2
Observation: AC2 = 13

Thought: Likewise the right endpoint is simply BC^2.
Action: Calculator
Action Input: BC2 = (Bx-Cx)**2 + (By-Cy)**2
Observation: BC2 = 73

Thought: inside_segment was True, so x0 is a valid candidate, so I evaluate f there too, using the unexpanded squared form from before.
Action: Calculator
Action Input: f_x0 = (x0-Cx)**2 + (a*(x0-Ax)+k)**2
Observation: f_x0 = 25/2

Thought: Take the min and max among all three surviving candidates.
Action: Calculator
Action Input: f_min = Min(AC2, BC2, f_x0)
Observation: f_min = 25/2

Action: Calculator
Action Input: f_max = Max(AC2, BC2, f_x0)
Observation: f_max = 73

Thought: m=sqrt(f_min), M=sqrt(f_max).
Action: Calculator
Action Input: m = sqrt(f_min)
Observation: m = 5*sqrt(2)/2

Action: Calculator
Action Input: M = sqrt(f_max)
Observation: M = sqrt(73)

Thought: Compute P=m+M.
Action: Calculator
Action Input: P = m + M
Observation: P = 5*sqrt(2)/2 + sqrt(73)

Thought: Eq() here is a numeric check, not a symbolic proof, so I test ALL FOUR options every time, never stopping early even once one comes back True. Read option A verbatim from the problem: "A. $P = \sqrt{13}+\sqrt{73}$"; A is sqrt(13)+sqrt(73). Test it.
Action: Calculator
Action Input: Eq(P, sqrt(13)+sqrt(73))
Observation: False

Thought: Not A. Read option B verbatim: "B. $P = \dfrac{5\sqrt{2}+2\sqrt{73}}{2}$"; B is (5*sqrt(2)+2*sqrt(73))/2. Test it.
Action: Calculator
Action Input: Eq(P, (5*sqrt(2)+2*sqrt(73))/2)
Observation: True

Thought: B came back True, but I still test C and D too, exactly as if none had matched yet. Read option C verbatim: "C. $P = 5\sqrt{2}+\sqrt{73}$"; C is 5*sqrt(2)+sqrt(73). Test it.
Action: Calculator
Action Input: Eq(P, 5*sqrt(2)+sqrt(73))
Observation: False

Thought: Not C. Read option D verbatim: "D. $P = \dfrac{5\sqrt{2}+\sqrt{73}}{2}$"; D is (5*sqrt(2)+sqrt(73))/2. Test it.
Action: Calculator
Action Input: Eq(P, (5*sqrt(2)+sqrt(73))/2)
Observation: False

Thought: Exactly one option came back True: B. That is the answer.
</think>
Final Answer: \boxed{B}
'''

# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] In your Thought, read off $A,B,C$ term by term (per PART 2). Store all SIX coordinates `Ax,Ay,Bx,By,Cx,Cy` and `K = <the given constant>` with the Calculator, one at a time; the $y$-coordinate of each point is just as easy to forget to store as the $x$-coordinate, so give it its own Thought and Calculator call too, never skip straight past it.
[Step 2] Compute `AB = sqrt((Bx-Ax)**2 + (By-Ay)**2)` and check `Eq(AB, K)`.
[Step 3] Compute `a = (By-Ay)/(Bx-Ax)`. This single result tells you which of the two cases from PART 2 you are in: an ordinary number means the MAIN CASE (steps 4-6 below); `zoo` means the VERTICAL CASE (steps 4v-6v below) - in that case do not retry this formula and do not invent a placeholder value, go directly to step 4v.

**MAIN CASE ($a$ was an ordinary number):**
[Step 4] Compute `k = Ay - Cy`. Never compute a $y$-intercept $b$.
[Step 5] Compute the critical point directly in closed form: `x0 = (Cx + a**2*Ax - a*k)/(a**2+1)`. Do NOT call `diff()` or `solve()` - the formula already IS the solved result.
[Step 6] Compute `inside_segment = (x0 - Min(Ax, Bx)) * (Max(Ax, Bx) - x0) >= 0` with ONE Calculator call (the product trick from PART 2) - never check the two bounds separately and combine them yourself. It is often `False` (do not assume $x_0$ is always inside the segment); only if `True` does `x0` become a candidate, via `f_x0 = (x0-Cx)**2 + (a*(x0-Ax)+k)**2` (compute this only when `inside_segment` is `True`).

**VERTICAL CASE (step 3 gave `zoo`) - skip steps 4-6 above entirely, do this instead:**
[Step 4v] There is no $k$ or $x_0$ in this case - do not try to compute them.
[Step 5v] Compute `inside_segment = (Cy - Min(Ay, By)) * (Max(Ay, By) - Cy) >= 0` - the same product trick as step 6, with $y$-coordinates instead of $x$-coordinates (PART 2's vertical case).
[Step 6v] Only if `inside_segment` is `True`, the third candidate is `f_x0 = (Ax-Cx)**2`.

**Both cases continue here:**
[Step 7] Compute the two endpoints DIRECTLY as squared distances, never by substituting into a line/quadratic: `AC2 = (Ax-Cx)**2 + (Ay-Cy)**2` and `BC2 = (Bx-Cx)**2 + (By-Cy)**2`.
[Step 8] Compute `f_min = Min(...)` and `f_max = Max(...)` over exactly the surviving candidates (`AC2`, `BC2`, and `f_x0` only if it was computed). Then `m = sqrt(f_min)` and `M = sqrt(f_max)`.
[Step 9] Compute whatever the problem asks (e.g. `P = m + M`). Go through ALL FOUR options, one at a time in order, and test EVERY one with `Eq(...)` - never stop early just because an earlier one already came back `True`; `Eq()` here is a high-precision numeric check, not a symbolic proof (as explained in the Calculator instructions above), so the discipline of checking all four every time is what keeps it trustworthy. For each option, quote its exact text from the problem verbatim in your Thought before assigning it a value. Exactly one of the four should come back `True`; that letter is your answer. Never pick the letter by eye.

Right before each Calculator call, write a Thought that spells out the EXACT line you are about to type (for example \"I will now type exactly: f_x0 = ...\"), then copy that line character-for-character into `Action Input` rather than retyping it from the earlier prose.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 50          # quy trinh day du ~18-19 luot tinh + toi da 4 luot doi chieu phuong an; nang tu 35 vi cau STT1069 (truong hop duong thang dung, hiem gap, chua co vi du mau) dung het 35 luot voi 25/35 la loi cu phap - can du cho nhieu lan thu-sai hon
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)